In [ ]:
###########################################################
# 과 제 명: [A2024_S05_2] 생활인구 분석 - 경기 부천시[31050]
# 파 일 명: S2.0.내부_DATA_to_DB.ipynb
# 작 성 자: 한영기
###########################################################
import os, importlib, warnings; warnings.filterwarnings(action='ignore')
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from config import config_A28       as config   ; importlib.reload(config)
from utils  import functions_utils    as fu       ; importlib.reload(fu)
from utils  import functions_log      as fl       ; importlib.reload(fl)
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from utils  import functions_lime     as flime    ; importlib.reload(flime)
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from utils  import functions_utils_db as fud      ; importlib.reload(fud)

from shapely          import wkt #함수는 문자열로 표현된 지오메트리를 파싱하여 실제 지오메트리 객체로 변환
from shapely.geometry import Polygon, mapping, shape,Point
import geopandas as gpd
#==================================================================================================
# [0-1] log 기록 정의
log = fl.Logger(config.PATH_LOG, config.LOG_FILE_NM ); log.set_log_level("info")
log.info("0-1 log 기록 정의")
#==================================================================================================
# [0-2] 실행 변수 정의
log.info("0.2.실행 변수 정의")
config_info = fu.f_get_config_info(config.config_run, config.ENV_DEV)
#==================================================================================================
# [0-3] 실행 변수
log.info("0.3.실행 변수 추가"); fu.f_memory_check()
# 전역 변수 선언 부분
# config_info["TODAY"]           = "20230630"

In [ ]:
# [분석 처리 시작]
###########################################################
# 모 듈 명: 모 듈 명 - S2.0.내부_DATA_to_DB.ipynb
# 파 일 명: S2.0.내부_DATA_to_DB.ipynb
# 작 성 자: 한영기
# 설    명: 설명
# 파일 버전: v1.0 [2024.07.01]
# 작성 일자: 2024.07.01
# 수정 일자: 2024.07.01 00:00 최초 작성
###########################################################
fl.f_log_line('[START]S2.0.내부_DATA_to_DB.ipynb')
# 1.실행 변수 정의
log.info("1.실행 변수 정의")
path_base         = "2024-06-14"

path_data_out     = os.path.join(config_info["PATH_DATA_COLLECT"],"01.외부",path_base)
path_data_in      = os.path.join(config_info["PATH_DATA_COLLECT"],"02.내부",path_base)
path_data_analyze = os.path.join(config_info["PATH_DATA_ANALYZE"])

path_database = os.path.join(config_info["PATH_DATA_OUTPUT_DATEBASE"],path_base)
path_pretreat = os.path.join(config_info["PATH_DATA_OUTPUT_PRETREAT"],path_base)
path_middle   = os.path.join(config_info["PATH_DATA_OUTPUT_MIDDLE"]  ,path_base)
path_result   = os.path.join(config_info["PATH_DATA_OUTPUT_RESULT"]  ,path_base)

---

In [ ]:
# [1.유동인구] : [DB] <- [csv]

In [ ]:
###########################################################################
# [1.1][DB][d10_유동인구_FLOWPOP] ← [T1_GG_50CELL_FLOWPOP_202201_4119.csv]
###########################################################################
# 1.처리기순
#   - from_data_type : [csv] -> [DB]
#   - file_encoding  : EUC-KR
#   - 데이터 좌표계    : 5178 # UTM-K
#   - 표준으로 변경처리 : 4326 # WGS84(경/위도)
###########################################################################
# 1.로그 변수 선언
log_text = ["1.1","[d10_유동인구_FLOWPOP]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
#path_data      = os.path.join(path_data_in,"1.유동인구")
path_data      = os.path.join('F:\\24Project\\A28_수원시')
path_file_list = os.path.join(path_data   ,"*\\T1_GG_50CELL_FLOWPOP*.csv")
# 3.대상테이블 ID
table_name = 'd10_유동인구_FLOWPOP'
# 4.대상테이블 생성 정보 SQL
sql_ddl       = f"""
CREATE TABLE `{table_name}` (
  `ADMI_CD` varchar(8) DEFAULT NULL,
  `CTY_NM`  varchar(20) DEFAULT NULL,
  `ADMI_NM` varchar(20) DEFAULT NULL,
  `UTMK_X` text DEFAULT NULL,
  `UTMK_Y` text DEFAULT NULL,
  `TIME_CD` varchar(2) DEFAULT NULL,
  `FORN_GB` text DEFAULT NULL,
  `M_10_CNT` text DEFAULT NULL,
  `M_15_CNT` text DEFAULT NULL,
  `M_20_CNT` text DEFAULT NULL,
  `M_25_CNT` text DEFAULT NULL,
  `M_30_CNT` text DEFAULT NULL,
  `M_35_CNT` text DEFAULT NULL,
  `M_40_CNT` text DEFAULT NULL,
  `M_45_CNT` text DEFAULT NULL,
  `M_50_CNT` text DEFAULT NULL,
  `M_55_CNT` text DEFAULT NULL,
  `M_60_CNT` text DEFAULT NULL,
  `M_65_CNT` text DEFAULT NULL,
  `M_70_CNT` text DEFAULT NULL,
  `F_10_CNT` text DEFAULT NULL,
  `F_15_CNT` text DEFAULT NULL,
  `F_20_CNT` text DEFAULT NULL,
  `F_25_CNT` text DEFAULT NULL,
  `F_30_CNT` text DEFAULT NULL,
  `F_35_CNT` text DEFAULT NULL,
  `F_40_CNT` text DEFAULT NULL,
  `F_45_CNT` text DEFAULT NULL,
  `F_50_CNT` text DEFAULT NULL,
  `F_55_CNT` text DEFAULT NULL,
  `F_60_CNT` text DEFAULT NULL,
  `F_65_CNT` text DEFAULT NULL,
  `F_70_CNT` text DEFAULT NULL,
  `ETL_YMD` varchar(8) DEFAULT NULL,
  `X` double DEFAULT NULL,
  `Y` double DEFAULT NULL,
  KEY `{table_name}_X_IDX` (`X`,`Y`,`ADMI_NM`) USING BTREE,
  KEY `{table_name}_ETL_YMD_IDX` (`ETL_YMD`,`ADMI_NM`,`ADMI_CD`,`CTY_NM`,`TIME_CD`) USING BTREE
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci;
"""
sql_index_ddl = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
# config_info["p_file_encoding"] = "EUC-KR" #[EUC-KR][UTF-8][utf-8-sig][ANSI]  파일의 encoding정보
config_info["p_table_name"]     = table_name
config_info["p_sql_ddl"]        = sql_ddl
#config_info["p_sql_index_ddl"] = sql_index_ddl
#config_info["p_column_cnt"]    = None # 202301부터 항목이 축소됨 [34->31]
config_info["p_coordinate"]     = ["UTMK_X","UTMK_Y","epsg:5178", "epsg:4326"]
#config_info["p_epsg_from"]     = None # UTM-K (GRS80)	EPSG:5179
#config_info["p_epsg_to"]       = None # WGS84 경/위도	EPSG:4326
#config_info["p_skiprows"]      = 1
###########################################################################
fud.f_csv_to_db2(log,log_text,config_info)
#[INFO   ][2024/06/15 18:49:01] 1.1.[끝][d10_유동인구T_FLOWPOP] [건수:188074885][처리시간:3:48:18]
#[INFO   ][2024/07/24 03:27:42] 1.1.[끝][d10_유동인구_FLOWPOP] [건수:188074885][처리시간:4:13:13]

---

In [ ]:
# [2.서비스인구] : [DB] <- [csv]

# d20_서비스인구_LIVEPOP 생성 SQL 공통
sql_ddl_common = f"""
CREATE TABLE `p_table_name` (
  `ADMI_CD` varchar(8) DEFAULT NULL,
  `CTY_NM`  varchar(100) DEFAULT NULL,
  `ADMI_NM` varchar(100) DEFAULT NULL,
  `UTMK_X`  double DEFAULT NULL,
  `UTMK_Y`  double DEFAULT NULL,
  `TIME_CD` varchar(2) DEFAULT NULL,
  `H_M_10_CNT` text DEFAULT NULL,
  `H_M_15_CNT` text DEFAULT NULL,
  `H_M_20_CNT` text DEFAULT NULL,
  `H_M_25_CNT` text DEFAULT NULL,
  `H_M_30_CNT` text DEFAULT NULL,
  `H_M_35_CNT` text DEFAULT NULL,
  `H_M_40_CNT` text DEFAULT NULL,
  `H_M_45_CNT` text DEFAULT NULL,
  `H_M_50_CNT` text DEFAULT NULL,
  `H_M_55_CNT` text DEFAULT NULL,
  `H_M_60_CNT` text DEFAULT NULL,
  `H_M_65_CNT` text DEFAULT NULL,
  `H_M_70_CNT` text DEFAULT NULL,
  `H_F_10_CNT` text DEFAULT NULL,
  `H_F_15_CNT` text DEFAULT NULL,
  `H_F_20_CNT` text DEFAULT NULL,
  `H_F_25_CNT` text DEFAULT NULL,
  `H_F_30_CNT` text DEFAULT NULL,
  `H_F_35_CNT` text DEFAULT NULL,
  `H_F_40_CNT` text DEFAULT NULL,
  `H_F_45_CNT` text DEFAULT NULL,
  `H_F_50_CNT` text DEFAULT NULL,
  `H_F_55_CNT` text DEFAULT NULL,
  `H_F_60_CNT` text DEFAULT NULL,
  `H_F_65_CNT` text DEFAULT NULL,
  `H_F_70_CNT` text DEFAULT NULL,
  `W_M_10_CNT` text DEFAULT NULL,
  `W_M_15_CNT` text DEFAULT NULL,
  `W_M_20_CNT` text DEFAULT NULL,
  `W_M_25_CNT` text DEFAULT NULL,
  `W_M_30_CNT` text DEFAULT NULL,
  `W_M_35_CNT` text DEFAULT NULL,
  `W_M_40_CNT` text DEFAULT NULL,
  `W_M_45_CNT` text DEFAULT NULL,
  `W_M_50_CNT` text DEFAULT NULL,
  `W_M_55_CNT` text DEFAULT NULL,
  `W_M_60_CNT` text DEFAULT NULL,
  `W_M_65_CNT` text DEFAULT NULL,
  `W_M_70_CNT` text DEFAULT NULL,
  `W_F_10_CNT` text DEFAULT NULL,
  `W_F_15_CNT` text DEFAULT NULL,
  `W_F_20_CNT` text DEFAULT NULL,
  `W_F_25_CNT` text DEFAULT NULL,
  `W_F_30_CNT` text DEFAULT NULL,
  `W_F_35_CNT` text DEFAULT NULL,
  `W_F_40_CNT` text DEFAULT NULL,
  `W_F_45_CNT` text DEFAULT NULL,
  `W_F_50_CNT` text DEFAULT NULL,
  `W_F_55_CNT` text DEFAULT NULL,
  `W_F_60_CNT` text DEFAULT NULL,
  `W_F_65_CNT` text DEFAULT NULL,
  `W_F_70_CNT` text DEFAULT NULL,
  `V_M_10_CNT` text DEFAULT NULL,
  `V_M_15_CNT` text DEFAULT NULL,
  `V_M_20_CNT` text DEFAULT NULL,
  `V_M_25_CNT` text DEFAULT NULL,
  `V_M_30_CNT` text DEFAULT NULL,
  `V_M_35_CNT` text DEFAULT NULL,
  `V_M_40_CNT` text DEFAULT NULL,
  `V_M_45_CNT` text DEFAULT NULL,
  `V_M_50_CNT` text DEFAULT NULL,
  `V_M_55_CNT` text DEFAULT NULL,
  `V_M_60_CNT` text DEFAULT NULL,
  `V_M_65_CNT` text DEFAULT NULL,
  `V_M_70_CNT` text DEFAULT NULL,
  `V_F_10_CNT` text DEFAULT NULL,
  `V_F_15_CNT` text DEFAULT NULL,
  `V_F_20_CNT` text DEFAULT NULL,
  `V_F_25_CNT` text DEFAULT NULL,
  `V_F_30_CNT` text DEFAULT NULL,
  `V_F_35_CNT` text DEFAULT NULL,
  `V_F_40_CNT` text DEFAULT NULL,
  `V_F_45_CNT` text DEFAULT NULL,
  `V_F_50_CNT` text DEFAULT NULL,
  `V_F_55_CNT` text DEFAULT NULL,
  `V_F_60_CNT` text DEFAULT NULL,
  `V_F_65_CNT` text DEFAULT NULL,
  `V_F_70_CNT` text DEFAULT NULL,
  `ETL_YMD` varchar(8) DEFAULT NULL,
  `X` double DEFAULT NULL,
  `Y` double DEFAULT NULL,
  KEY `p_table_name_ETL_YMD_IDX` (`ETL_YMD`,`ADMI_NM`,`ADMI_CD`,`CTY_NM`,`TIME_CD`) USING BTREE,
  KEY `p_table_name_X_IDX` (`X`,`Y`,`ADMI_CD`) USING BTREE
)
"""

In [ ]:
###########################################################################
# [2.2][DB][d20_서비스인구_LIVEPOP_2023] ← [T3_GG_50CELL_LIVEPOP_202301_4119.csv]
###########################################################################
# 1.처리기순
#   - from_data_type : [csv] -> [DB]
#   - file_encoding  : EUC-KR
#   - 데이터 좌표계    : 5178 # UTM-K
#   - 표준으로 변경처리 : 4326 # WGS84(경/위도)
###########################################################################
# 1.로그 변수 선언
log_text = ["2.2","[d20_서비스인구_LIVEPOP_2023]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
#path_data      = os.path.join(path_data_in,"2.서비스인구","2023")
path_data      = os.path.join('F:\\24Project\\A28_수원시')
path_file_list = os.path.join(path_data   ,"*\\T3_GG_50CELL_LIVEPOP_2023*.csv")
# 3.대상테이블 ID
table_name = 'd20_서비스인구_LIVEPOP_2023'
# 4.대상테이블 생성 정보 SQL
sql_ddl = sql_ddl_common.replace('p_table_name',table_name)
sql_index_ddl = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
# config_info["p_file_encoding"]= "EUC-KR" #[EUC-KR][UTF-8][utf-8-sig][ANSI]  파일의 encoding정보
config_info["p_table_name"]     = table_name
config_info["p_sql_ddl"]        = sql_ddl
#config_info["p_sql_index_ddl"] = sql_index_ddl
#config_info["p_column_cnt"]    = None # 202301부터 항목이 축소됨 [34->31]
config_info["p_coordinate"]     = ["UTMK_X","UTMK_Y","epsg:5178", "epsg:4326"]
#config_info["p_epsg_from"]     = None # UTM-K (GRS80)	EPSG:5179
#config_info["p_epsg_to"]       = None # WGS84 경/위도	EPSG:4326
#config_info["p_skiprows"]      = 1
###########################################################################
fud.f_csv_to_db2(log,log_text,config_info)
#[INFO   ][2024/06/16 15:16:16] 1.2.[끝][d20_서비스인구_LIVEPOP_2023] [건수:119471493][처리시간:6:17:24]
#[INFO   ][2024/07/26 00:48:26] 2.2.[끝][d20_서비스인구_LIVEPOP_2023] [건수:119471493][처리시간:6:28:18]

In [ ]:
###########################################################################
# [2.3][DB][d20_서비스인구_LIVEPOP_2024] ← [T3_GG_50CELL_LIVEPOP_202401_4119.csv]
###########################################################################
# 1.처리기순
#   - from_data_type : [csv] -> [DB]
#   - file_encoding  : EUC-KR
#   - 데이터 좌표계    : 5178 # UTM-K
#   - 표준으로 변경처리 : 4326 # WGS84(경/위도)
###########################################################################
# 1.로그 변수 선언
log_text = ["1.3","[d20_서비스인구_LIVEPOP_2024]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
#path_data      = os.path.join(path_data_in,"2.서비스인구","2024")
path_data      = os.path.join('F:\\24Project\\A28_수원시')
path_file_list = os.path.join(path_data   ,"*\\T3_GG_50CELL_LIVEPOP_2024*.csv")
# 3.대상테이블 ID
table_name = 'd20_서비스인구_LIVEPOP_2024'
# 4.대상테이블 생성 정보 SQL
sql_ddl = sql_ddl_common.replace('p_table_name',table_name)
sql_index_ddl = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
# config_info["p_file_encoding"]= "EUC-KR" #[EUC-KR][UTF-8][utf-8-sig][ANSI]  파일의 encoding정보
config_info["p_table_name"]     = table_name
config_info["p_sql_ddl"]       = sql_ddl
#config_info["p_sql_index_ddl"] = sql_index_ddl
#config_info["p_column_cnt"]    = None # 202301부터 항목이 축소됨 [34->31]
config_info["p_coordinate"]     = ["UTMK_X","UTMK_Y","epsg:5178", "epsg:4326"]
#config_info["p_epsg_from"]     = None # UTM-K (GRS80)	EPSG:5179
#config_info["p_epsg_to"]       = None # WGS84 경/위도	EPSG:4326
#config_info["p_skiprows"]      = 1
###########################################################################
fud.f_csv_to_db2(log,log_text,config_info)
#[INFO   ][2024/06/16 17:21:11] 1.3.[끝][d20_서비스인구_LIVEPOP_2024] [건수:39012835][처리시간:2:04:47]
#[INFO   ][2024/07/24 05:30:17] 1.3.[끝][d20_서비스인구_LIVEPOP_2024] [건수:39012835][처리시간:2:02:33]

---

In [ ]:
# [3.카드] : [DB] <- [csv]

In [ ]:
###########################################################################
# [3.0][DB][d30_카드사업종] ← [NICE평가정보_카드사 업종코드_240522.xlsx]
###########################################################################
# 1.처리기순
#   - from_data_type : [csv] -> [DB]
###########################################################################
# 1.로그 변수 선언
log_text = ["3.0","[d30_카드사업종]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
path_data      = os.path.join(path_data_in,"0.기준")
path_file_list = os.path.join(path_data   ,"NICE평가정보_카드사 업종코드_240522.xlsx")
# 3.대상테이블 ID
table_name     = 'd30_카드사업종'
# 4.대상테이블 생성 정보 SQL
sql_ddl        = f""""""
sql_index_ddl  = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
config_info["p_table_name"]     = table_name
###########################################################################
fud.f_excel_to_db2(log,log_text,config_info)
#[INFO   ][2024/07/24 22:32:21] 3.0.[종료][d30_카드사업종] [처리건수:426건] [처리시간:0:00:00]

In [ ]:
###########################################################################
# [3.1][DB][d31_카드_가맹점_mer] ← [tbsh_gyeonggi_mer_c_202201.csv]
###########################################################################
# 1.처리기순
#   - from_data_type : [csv] -> [DB]
###########################################################################
# 1.로그 변수 선언
log_text = ["3.1","[d31_카드_가맹점_mer]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
path_data      = os.path.join(path_data_in,"3.카드","1.가맹점 정보 데이터")
path_file_list = os.path.join(path_data   ,"*.csv")
# 3.대상테이블 ID
table_name = 'd31_카드_가맹점_mer'
# 4.대상테이블 생성 정보 SQL
sql_ddl = f"""
CREATE TABLE `{table_name}` (
  `ta_ym` varchar(6) DEFAULT NULL,
  `cty_rgn_no` varchar(10) DEFAULT NULL,
  `admi_city_no` varchar(10) DEFAULT NULL,
  `blk_cd` varchar(10) DEFAULT NULL,
  `card_tpbuz_cd` varchar(10) DEFAULT NULL,
  `card_tpbuz_nm_1` varchar(100) DEFAULT NULL,
  `card_tpbuz_nm_2` varchar(100) DEFAULT NULL,
  `card_tpbuz_nm_3` varchar(100) DEFAULT NULL,
  `sale` text DEFAULT NULL,
  `mm_cnt` text DEFAULT NULL,
  `mer_cnt` text DEFAULT NULL,
  `fran_cnt` text DEFAULT NULL,
  `open_cnt` text DEFAULT NULL,
  `stop_cnt` text DEFAULT NULL,
  `close_cnt` text DEFAULT NULL,
  `open_11_cnt` text DEFAULT NULL,
  `open_12_23_cnt` text DEFAULT NULL,
  `open_24_35_cnt` text DEFAULT NULL,
  `open_36_47_cnt` text DEFAULT NULL,
  `open_48_59_cnt` text DEFAULT NULL,
  `open_60_cnt` text DEFAULT NULL,
  KEY `{table_name}_ta_ym_IDX` (`ta_ym`,`admi_city_no`,`card_tpbuz_nm_1`,`card_tpbuz_nm_2`,`card_tpbuz_nm_3`) USING BTREE
)
"""
sql_index_ddl = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
config_info["p_table_name"]     = table_name
config_info["p_sql_ddl"]        = sql_ddl
###########################################################################
fud.f_csv_to_db2(log,log_text,config_info)
#[INFO   ][2024/07/27 01:11:18] 3.1.[끝][d31_카드_가맹점_mer] [건수:1396560][처리시간:0:01:05]

In [ ]:
###########################################################################
# [3.2][DB][d32_카드_성_연령_agrde] ← [tbsh_gyeonggi_agrde_202201.csv]
###########################################################################
# 성별/연령별 카드 매출 데이터
#   - from_data_type : [csv] -> [DB]
###########################################################################
# 1.로그 변수 선언
log_text = ["3.2","[d32_카드_성_연령_agrde]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
path_data      = os.path.join(path_data_in,"3.카드","2.매출_성_연령")
path_file_list = os.path.join(path_data   ,"*.csv")
# 3.대상테이블 ID
table_name     = 'd32_카드_성_연령_agrde'
# 4.대상테이블 생성 정보 SQL
sql_ddl = f"""
CREATE TABLE `{table_name}` (
  `ta_ymd` varchar(8) DEFAULT NULL,
  `cty_rgn_no` varchar(10) DEFAULT NULL,
  `admi_cty_no` varchar(10) DEFAULT NULL,
  `blk_cd`      varchar(10) NULL,
  `card_tpbuz_cd` varchar(10) DEFAULT NULL,
  `card_tpbuz_nm_1` varchar(100) DEFAULT NULL,
  `card_tpbuz_nm_2` varchar(100) DEFAULT NULL,
  `card_tpbuz_nm_3` varchar(100) DEFAULT NULL,
  `sale` text DEFAULT NULL,
  `hour` varchar(4) DEFAULT NULL,
  `sex` varchar(4) DEFAULT NULL,
  `age` varchar(4) DEFAULT NULL,
  `cnt` text DEFAULT NULL,
  `amt` text DEFAULT NULL,
  KEY `{table_name}_ta_ymd_IDX_1` (`ta_ymd`,`admi_cty_no`,`card_tpbuz_nm_1`,`card_tpbuz_nm_2`,`hour`,`sex`,`age`) USING BTREE,
  KEY `{table_name}_admi_cty_no_IDX` (`admi_cty_no`,`hour`,`sex`,`age`) USING BTREE
)
"""
sql_index_ddl = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
config_info["p_table_name"]     = table_name
config_info["p_sql_ddl"]        = sql_ddl
###########################################################################
fud.f_csv_to_db2(log,log_text,config_info)
#[INFO   ][2024/06/16 20:58:31] 3.2.[끝][d32_카드_성_연령_agrde] [건수:131957762][처리시간:1:14:16]
#[INFO   ][2024/07/27 02:35:32] 3.2.[끝][d32_카드_성_연령_agrde] [건수:131957762][처리시간:1:24:14]

In [ ]:
###########################################################################
# [3.3][DB][d33_카드_시간대_selng] ← [tbsh_gyeonggi_selng_202201.csv]
###########################################################################
# 가맹점ㆍ시간대별 카드 매출 통계 데이터
#   - from_data_type : [csv] -> [DB]
###########################################################################
# 1.로그 변수 선언
log_text = ["3.3","[d33_카드_시간대_selng]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
path_data      = os.path.join(path_data_in,"3.카드","3.매출_시간대")
path_file_list = os.path.join(path_data   ,"*.csv")
# 3.대상테이블 ID
table_name = 'd33_카드_시간대_selng'
# 4.대상테이블 생성 정보 SQL
sql_ddl = f"""
CREATE TABLE `{table_name}` (
  `ta_ymd` varchar(8) DEFAULT NULL,
  `cty_rgn_no` varchar(10) DEFAULT NULL,
  `admi_cty_no` varchar(10) DEFAULT NULL,
  `blk_cd` varchar(10) DEFAULT NULL,
  `card_tpbuz_cd` varchar(10) DEFAULT NULL,
  `card_tpbuz_nm_1` varchar(100) DEFAULT NULL,
  `card_tpbuz_nm_2` varchar(100) DEFAULT NULL,
  `card_tpbuz_nm_3` varchar(100) DEFAULT NULL,
  `sale` text DEFAULT NULL,
  `total_cnt` text DEFAULT NULL,
  `total_amt` text DEFAULT NULL,
  `indv_max_amt` text DEFAULT NULL,
  `indv_max_cnt` text DEFAULT NULL,
  `indv_min_amt` text DEFAULT NULL,
  `indv_min_cnt` text DEFAULT NULL,
  `indv_med_amt` text DEFAULT NULL,
  `indv_med_cnt` text DEFAULT NULL,
  `entp_max_amt` text DEFAULT NULL,
  `entp_max_cnt` text DEFAULT NULL,
  `entp_min_amt` text DEFAULT NULL,
  `entp_min_cnt` text DEFAULT NULL,
  `entp_med_amt` text DEFAULT NULL,
  `entp_med_cnt` text DEFAULT NULL,
  `hour` varchar(4) DEFAULT NULL,
  `indv_cnt` text DEFAULT NULL,
  `indv_amt` text DEFAULT NULL,
  `entp_cnt` text DEFAULT NULL,
  `entp_amt` text DEFAULT NULL,
  `fran_cnt` text DEFAULT NULL,
  `fran_amt` text DEFAULT NULL,
  KEY `{table_name}_ta_ymd_IDX_1` (`ta_ymd`,`admi_cty_no`,`card_tpbuz_nm_1`,`card_tpbuz_nm_2`,`hour`) USING BTREE,
  KEY `{table_name}_admi_cty_no_IDX` (`admi_cty_no`,`hour`) USING BTREE
)
"""
sql_index_ddl = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
config_info["p_table_name"]     = table_name
config_info["p_sql_ddl"]        = sql_ddl
###########################################################################
fud.f_csv_to_db2(log,log_text,config_info)
#[INFO   ][2024/06/16 18:51:28] 3.3.[끝][d33_카드_시간대_selng] [건수:50699295][처리시간:0:55:13]
#[INFO   ][2024/07/27 03:43:34] 3.3.[끝][d33_카드_시간대_selng] [건수:50699295][처리시간:0:56:16]

In [ ]:
###########################################################################
# [3.4][DB][d34_카드_유입지_성연령] ← [tbsh_gyeonggi_day_inflow_202201.csv]
###########################################################################
# 유입지별 매출 데이터
#   - from_data_type : [csv] -> [DB]
###########################################################################
# 1.로그 변수 선언
log_text = ["3.4","[d34_카드_유입지_성연령]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
path_data      = os.path.join(path_data_in,"3.카드","4.매출_유입지_성_연령")
path_file_list = os.path.join(path_data   ,"*.csv")
# 3.대상테이블 ID
table_name = 'd34_카드_유입지_성연령'
# 4.대상테이블 생성 정보 SQL
sql_ddl = f"""
CREATE TABLE `{table_name}` (
  `ta_ymd` varchar(8) DEFAULT NULL,
  `cty_rgn_no` varchar(10) DEFAULT NULL,
  `admi_cty_no` varchar(10) DEFAULT NULL,
  `card_tpbuz_cd` varchar(10) DEFAULT NULL,
  `card_tpbuz_nm_1` varchar(100) DEFAULT NULL,
  `card_tpbuz_nm_2` varchar(100) DEFAULT NULL,
  `card_tpbuz_nm_3` varchar(100) DEFAULT NULL,
  `sale` text DEFAULT NULL,
  `inflow_cd` varchar(10) DEFAULT NULL,
  `sex` varchar(4) DEFAULT NULL,
  `age` varchar(4) DEFAULT NULL,
  `day` varchar(4) DEFAULT NULL,
  `amt` text DEFAULT NULL,
  `cnt` text DEFAULT NULL,
  KEY `{table_name}_ta_ymd_IDX_1` (`ta_ymd`,`admi_cty_no`,`card_tpbuz_nm_1`,`card_tpbuz_nm_2`,`sex`,`age`,`day`,`inflow_cd`) USING BTREE,
  KEY `{table_name}_admi_cty_no_IDX` (`admi_cty_no`,`inflow_cd`,`sex`,`age`,`day`) USING BTREE

)
"""
sql_index_ddl = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
config_info["p_table_name"]     = table_name
config_info["p_sql_ddl"]        = sql_ddl
###########################################################################
fud.f_csv_to_db2(log,log_text,config_info)
#[INFO   ][2024/06/16 19:38:55] 3.4.[끝][d34_카드_유입지_성연령] [건수:84387077][처리시간:0:47:26]
#[INFO   ][2024/07/27 04:38:50] 3.4.[끝][d34_카드_유입지_성연령] [건수:84387077][처리시간:0:55:15]

---

In [ ]:
# [4.신용] : [DB] <- [csv]

In [ ]:
###########################################################################
# [4.1][DB][d41_신용_stat] ← [GYEONGGI_CELL_STAT_202201.csv]
###########################################################################
# 1.처리기순
#   - from_data_type : [csv] -> [DB]
###########################################################################
# 1.로그 변수 선언
log_text = ["4.1","[d41_신용_stat]"]; fu.f_line(); log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.내부데이터 대상 폴드
path_data      = os.path.join(path_data_in,"4.신용")
path_file_list = os.path.join(path_data   ,"*.csv")
# 3.대상테이블 ID
table_name = 'd41_신용_stat'
# 4.대상테이블 생성 정보 SQL
sql_ddl       = f"""
CREATE TABLE `{table_name}` (
  `BS_YR_MON` varchar(6) DEFAULT NULL,
  `CELL_100` text DEFAULT NULL,
  `TOT_REG_CD` text DEFAULT NULL,
  `hcd`    varchar(10) DEFAULT NULL,
  `GENDER` varchar(2) DEFAULT NULL,
  `AGE`    varchar(2) DEFAULT NULL,
  `TOT_CNT` text DEFAULT NULL,
  `M_CNT` text DEFAULT NULL,
  `FM_CNT` text DEFAULT NULL,
  `YES_INCOME` text DEFAULT NULL,
  `SELF_ENT` text DEFAULT NULL,
  `ETC_CNT` text DEFAULT NULL,
  `FOR_CNT` text DEFAULT NULL,
  `AVG_DIST` text DEFAULT NULL,
  `HIGH_CNT` text DEFAULT NULL,
  `AVG_INC` text DEFAULT NULL,
  `AVG_INC_1` text DEFAULT NULL,
  `AVG_INC_2` text DEFAULT NULL,
  `AVG_INC_3` text DEFAULT NULL,
  `AVG_INC_4` text DEFAULT NULL,
  `AVG_INC_5` text DEFAULT NULL,
  `AVG_INC_6` text DEFAULT NULL,
  `AVG_INC_7` text DEFAULT NULL,
  `AVG_INC_8` text DEFAULT NULL,
  `AVG_INC_9` text DEFAULT NULL,
  `AVG_INC_10` text DEFAULT NULL,
  `MED_INC` text DEFAULT NULL,
  `YES_HOUS_CNT` text DEFAULT NULL,
  `AVG_EVHOUS1` text DEFAULT NULL,
  `CAR_CNT` text DEFAULT NULL,
  `ABROCAR_CNT` text DEFAULT NULL,
  `CAR_NEW_CNT` text DEFAULT NULL,
  `AVG_CAR` text DEFAULT NULL,
  `AVG_EC_CAR` text DEFAULT NULL,
  `CARD_OWN_CNT` text DEFAULT NULL,
  `CRED_OWN_CNT` text DEFAULT NULL,
  `AVG_CRED_OWN` text DEFAULT NULL,
  `CHECK_OWN_CNT` text DEFAULT NULL,
  `AVG_CHECK_OWN` text DEFAULT NULL,
  `AVG_CARD` text DEFAULT NULL,
  `AVG_SELLING_CRED` text DEFAULT NULL,
  `AVG_CASH` text DEFAULT NULL,
  `AVG_LUMP` text DEFAULT NULL,
  `AVG_INST` text DEFAULT NULL,
  `AVG_CRED` text DEFAULT NULL,
  `AVG_CHECK` text DEFAULT NULL,
  `AVG_CRED_ABRO` text DEFAULT NULL,
  `AVG_CASH_ABRO` text DEFAULT NULL,
  `DELQ_S_CNT` text DEFAULT NULL,
  `DELQ_L_CNT` text DEFAULT NULL,
  `LOAN_CNT` text DEFAULT NULL,
  `LOAN1_CNT` text DEFAULT NULL,
  `LOAN2_CNT` text DEFAULT NULL,
  `CRED_LOAN_CNT` text DEFAULT NULL,
  `HOUS_LOAN_CNT` text DEFAULT NULL,
  `BANK_LOAN_CNT` text DEFAULT NULL,
  `POL_LOAN_CNT` text DEFAULT NULL,
  `AVG_LOAN` text DEFAULT NULL,
  `AVG_LOAN1` text DEFAULT NULL,
  `AVG_LOAN2` text DEFAULT NULL,
  `AVG_CRED_LOAN` text DEFAULT NULL,
  `AVG_HOUS_LOAN` text DEFAULT NULL,
  `CARD_LOAN_CNT` text DEFAULT NULL,
  `AVG_CARD_LOAN` text DEFAULT NULL,
  `LOAN_LIMIT_CNT` text DEFAULT NULL,
  `AVG_LOAN_LIMIT` text DEFAULT NULL,
  `NEW_LOAN_CNT` text DEFAULT NULL,
  `NEW_LOAN1_CNT` text DEFAULT NULL,
  `NEW_LOAN2_CNT` text DEFAULT NULL,
  `NEW_CRED_CNT` text DEFAULT NULL,
  `NEW_HOUS_CNT` text DEFAULT NULL,
  `AVG_SCORE` text DEFAULT NULL,
  `CNT_1` text DEFAULT NULL,
  `CNT_2` text DEFAULT NULL,
  `CNT_3` text DEFAULT NULL,
  `CNT_4` text DEFAULT NULL,
  `CNT_5` text DEFAULT NULL,
  `CNT_6` text DEFAULT NULL,
  `CNT_7` text DEFAULT NULL,
  `CNT_8` text DEFAULT NULL,
  `CNT_9` text DEFAULT NULL,
  `CNT_10` text DEFAULT NULL,
  `CONSUME_CNT` text DEFAULT NULL,
  `RAT_BIG_ENT` text DEFAULT NULL,
  `RAT_YESINCOME` text DEFAULT NULL,
  `RAT_PROENT` text DEFAULT NULL,
  `RAT_OWNER` text DEFAULT NULL,
  `RAT_RUN` text DEFAULT NULL,
  `RAT_PRORUN` text DEFAULT NULL,
  `RAT_ETC` text DEFAULT NULL,
  `APT_CNT` text DEFAULT NULL,
  `NAPT_CNT` text DEFAULT NULL,
  `OWN_HOUS_CNT` text DEFAULT NULL,
  `NOWN_HOUS_CNT` text DEFAULT NULL,
  `PYN_U20_CNT` text DEFAULT NULL,
  `PYN_O20_CNT` text DEFAULT NULL,
  `PYN_O30_CNT` text DEFAULT NULL,
  `PYN_O40_CNT` text DEFAULT NULL,
  `RAT_INC_SALE` text DEFAULT NULL,
  `RAT_INC_RENT` text DEFAULT NULL,
  `AVG_HIGH_YINC` text DEFAULT NULL,
  `RAT_YINC_1` text DEFAULT NULL,
  `RAT_YINC_2` text DEFAULT NULL,
  `RAT_YINC_3` text DEFAULT NULL,
  `RAT_YINC_4` text DEFAULT NULL,
  `RAT_YINC_5` text DEFAULT NULL,
  `RAT_YINC_6` text DEFAULT NULL,
  `RAT_YINC_7` text DEFAULT NULL,
  KEY `{table_name}_BS_YR_MON_IDX` (`BS_YR_MON`,`hcd`,`GENDER`,`AGE`) USING BTREE
)
"""
sql_index_ddl = f""""""
###########################################################################
config_info["p_path_data"]      = path_data
config_info["p_path_file_list"] = path_file_list
config_info["p_table_name"]     = table_name
config_info["p_sql_ddl"]        = sql_ddl
# config_info["sql_index_ddl"]    = sql_index_ddl
###########################################################################
fud.f_csv_to_db2(log,log_text,config_info)
#[INFO   ][2024/07/27 04:43:55] 4.1.[끝][d41_신용_stat] [건수:1686588][처리시간:0:05:05]

---